<a href="https://colab.research.google.com/github/tanwarAalok/airtribe-assignments/blob/main/LLM_Powered_File_System_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
!pip install google-genai PyPDF2 python-docx

In [39]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [40]:
%%writefile fs_tools.py
import os
import datetime
from pathlib import Path
import PyPDF2
import docx

def read_file(filepath: str) -> dict:

    print(f"\n[System Tool Executed] 🔍 read_file(filepath='{filepath}')")

    path = Path(filepath)
    if not path.exists():
        return {"error": f"File not found: {filepath}", "status": "failed"}

    metadata = {
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
    }
    content = ""

    try:
        if metadata["extension"] == ".txt":
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
        elif metadata["extension"] == ".pdf":
            with open(path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                content = "\n".join(page.extract_text() for page in reader.pages if page.extract_text())
        elif metadata["extension"] == ".docx":
            doc = docx.Document(path)
            content = "\n".join(paragraph.text for paragraph in doc.paragraphs)
        else:
            return {"error": f"Unsupported file type: {metadata['extension']}", "status": "failed"}

        return {"status": "success", "content": content, "metadata": metadata}
    except Exception as e:
        return {"error": str(e), "status": "failed"}

def list_files(directory: str, extension: str = "") -> list:

    print(f"\n[System Tool Executed] 📂 list_files(directory='{directory}', extension='{extension}')")

    path = Path(directory)
    if not path.exists() or not path.is_dir():
        return [{"error": f"Directory not found: {directory}", "status": "failed"}]

    files_info = []
    try:
        for item in path.iterdir():
            if item.is_file():
                if extension and not item.suffix.lower() == extension.lower():
                    continue

                stat = item.stat()
                mod_time = datetime.datetime.fromtimestamp(stat.st_mtime).isoformat()
                files_info.append({
                    "name": item.name,
                    "path": str(item),
                    "size_bytes": stat.st_size,
                    "modified_date": mod_time
                })
        return files_info
    except Exception as e:
        return [{"error": str(e), "status": "failed"}]

def write_file(filepath: str, content: str) -> dict:

    print(f"\n[System Tool Executed] ✏️ write_file(filepath='{filepath}')")

    path = Path(filepath)
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        return {"status": "success", "message": f"Successfully wrote to {filepath}"}
    except Exception as e:
        return {"status": "failed", "error": str(e)}

def search_in_file(filepath: str, keyword: str) -> dict:

    print(f"\n[System Tool Executed] 🔎 search_in_file(filepath='{filepath}', keyword='{keyword}')")

    read_result = read_file(filepath)
    if read_result.get("status") == "failed":
        return read_result

    content = read_result.get("content", "")
    keyword_lower = keyword.lower()
    content_lower = content.lower()

    matches = []
    start = 0
    context_window = 40

    while True:
        idx = content_lower.find(keyword_lower, start)
        if idx == -1:
            break

        context_start = max(0, idx - context_window)
        context_end = min(len(content), idx + len(keyword) + context_window)

        snippet = content[context_start:context_end].strip()
        matches.append(f"...{snippet}...")

        start = idx + len(keyword)

    return {
        "status": "success",
        "keyword": keyword,
        "matches_found": len(matches),
        "context_snippets": matches
    }

Overwriting fs_tools.py


In [41]:
%%writefile generate_dummies.py
import os

def create_dummy_data():
    os.makedirs("resumes", exist_ok=True)

    resumes = {
        "resume_john_doe.txt": "John Doe\nSoftware Engineer\nSkills: Python, Django, AWS, SQL\nExperience: 5 years building scalable web applications in Python.",
        "resume_jane_smith.txt": "Jane Smith\nData Scientist\nSkills: R, SQL, Machine Learning, Tableau\nExperience: 3 years analyzing retail datasets.",
        "resume_alice_jones.txt": "Alice Jones\nDevOps Engineer\nSkills: Docker, Kubernetes, CI/CD, Python\nExperience: 4 years automating deployment pipelines.",
        "resume_bob_brown.txt": "Bob Brown\nFrontend Developer\nSkills: HTML, CSS, JavaScript, React\nExperience: 2 years building UI components.",
        "resume_charlie_davis.txt": "Charlie Davis\nBackend Developer\nSkills: Java, Spring Boot, MySQL, Python scripting\nExperience: 6 years writing microservices."
    }

    for filename, content in resumes.items():
        filepath = os.path.join("resumes", filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"Created: {filepath}")

if __name__ == "__main__":
    create_dummy_data()

Overwriting generate_dummies.py


In [42]:
%%writefile llm_file_assistant.py
from google import genai
from google.genai import types
import fs_tools

client = genai.Client()

def run_assistant(user_query: str):
    print(f"\nUser: {user_query}")
    print("-" * 50)

    tools = [
        fs_tools.list_files,
        fs_tools.read_file,
        fs_tools.search_in_file,
        fs_tools.write_file
    ]

    chat = client.chats.create(
        model="models/gemini-3.1-flash-lite",
        config=types.GenerateContentConfig(
            system_instruction="You are an autonomous file system assistant. Use your tools to explore folders and parse files. If summarizing, use the write_file tool to save it.",
            temperature=0.2,
            tools=tools
        )
    )


    response = chat.send_message(user_query)

    print(f"\nAssistant:\n{response.text}")
    print("=" * 80)

if __name__ == "__main__":
    # Test Queries
    queries = [
        "List all the text files in the './resumes' folder.",
        "Find resumes mentioning 'Python' experience in the './resumes' folder.",
        "Read 'resume_john_doe.txt' and create a summary file for him called 'john_summary.txt'."
    ]

    for q in queries:
        run_assistant(q)

Overwriting llm_file_assistant.py


In [43]:
!python generate_dummies.py

Created: resumes/resume_john_doe.txt
Created: resumes/resume_jane_smith.txt
Created: resumes/resume_alice_jones.txt
Created: resumes/resume_bob_brown.txt
Created: resumes/resume_charlie_davis.txt


In [44]:
!python llm_file_assistant.py


User: List all the text files in the './resumes' folder.
--------------------------------------------------

[System Tool Executed] 📂 list_files(directory='./resumes', extension='txt')

[System Tool Executed] 📂 list_files(directory='./resumes', extension='')

Assistant:
The text files in the `./resumes` folder are:

*   `resume_charlie_davis.txt`
*   `resume_bob_brown.txt`
*   `resume_jane_smith.txt`
*   `resume_john_doe.txt`
*   `resume_alice_jones.txt`

User: Find resumes mentioning 'Python' experience in the './resumes' folder.
--------------------------------------------------

[System Tool Executed] 📂 list_files(directory='./resumes', extension='')

[System Tool Executed] 🔎 search_in_file(filepath='resumes/resume_charlie_davis.txt', keyword='Python')

[System Tool Executed] 🔍 read_file(filepath='resumes/resume_charlie_davis.txt')

[System Tool Executed] 🔎 search_in_file(filepath='resumes/resume_bob_brown.txt', keyword='Python')

[System Tool Executed] 🔍 read_file(filepath='resume